# 02. Literały definiowane przez użytkownika

## Cele

Po tym notebooku potrafisz:

* zdefiniować operator literałowy `operator""_sufiks` i umieścić go w przestrzeni nazw `…::literals`;
* wyjaśnić, które listy parametrów są dozwolone i którą formę operatora kompilator wybierze dla danego literału;
* zbudować typ z jednostką (*strong type*) i literały, które go tworzą;
* napisać funkcję przyjmującą dowolny czas trwania `std::chrono` i łączyć własne literały z literałami biblioteki.

## Po co własne literały?

Liczba `double` albo `int` nie mówi, w jakiej jest jednostce. Funkcja `void reserve(std::size_t size)` przyjmie
zarówno bajty, jak i kilobajty, a kompilator nie zauważy pomyłki. Pablo Halpern nazywa taki typ „typem-zlewem”:
jako parametr pasuje do wielu niezwiązanych ze sobą znaczeń
([Halpern, *Embracing User Defined Literals Safely*, ACCU 2022](https://accu.org/conf-docs/PDFs_2022/pablo_halpern_ACCU-2022-Embracing-User-Defined-Literals.pdf)).

Rozwiązaniem jest **typ z jednostką** (*strong type*): mała struktura, która opakowuje wartość i nie zamienia się
niejawnie w liczbę ani z liczby
([Fluent C++: Strong types for strong interfaces](https://www.fluentcpp.com/2016/12/08/strong-types-for-strong-interfaces/)).
Literał użytkownika sprawia, że taki typ jest wygodny: `4_kib` czyta się jak zapis w dokumentacji,
a wartość ma właściwy typ od pierwszej chwili. Core Guidelines pokazują tę ideę na przykładzie prędkości:
`change_speed(23_m / 10s)` zamiast `change_speed(2.3)`
([C++ Core Guidelines P.1](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#rp-direct)).

## Operator literałowy

Literał użytkownika to literał zakończony sufiksem, który zdefiniowaliśmy sami. Za sufiksem stoi funkcja —
**operator literałowy** — o nazwie `operator""_sufiks`
([cppreference: User-defined literals](https://en.cppreference.com/cpp/language/user_literal#Literal_operators)):

```cpp
#include <cstdint>
#include <iostream>

struct Bytes {
    std::uint64_t value;
};

constexpr Bytes operator""_kib(unsigned long long n) {
    return Bytes{n * 1024};
}

int main() {
    Bytes buffer = 4_kib;  // the compiler calls operator""_kib(4ULL)
    std::cout << buffer.value << '\n';  // 4096

    Bytes same = operator""_kib(4);  // an explicit call works too
    std::cout << same.value << '\n';  // 4096
}
```

Kilka zasad:

* **Sufiks musi zaczynać się od `_`.** Sufiksy bez podkreślenia są zarezerwowane dla biblioteki standardowej,
  a sufiksy z podwójnym podkreśleniem `__` — dla implementacji
  ([standard: usrlit.suffix](https://eel.is/c++draft/usrlit.suffix#1)). W tym laboratorium sufiksy piszemy
  małymi literami (`_kib`), tak jak sufiksy biblioteki standardowej.
* **Bez spacji po `""`.** Starszy zapis `operator"" _kib` (ze spacją) jest przestarzały
  ([CWG 2521](https://cplusplus.github.io/CWG/issues/2521.html), C++23) — część starszych artykułów i stron
  wciąż go używa.
* **Operator literałowy to zwykła funkcja**: może być `constexpr`, można go wywołać jawnie, a typ wyniku jest
  dowolny ([cppreference: Literal operators](https://en.cppreference.com/cpp/language/user_literal#Literal_operators)).

## Forma całkowita: `unsigned long long`

Operator z parametrem `unsigned long long` obsługuje **każdy** literał całkowity z tym sufiksem: w dowolnym
systemie liczbowym i z separatorami cyfr. Kompilator najpierw oblicza wartość literału (bez sufiksu),
a potem przekazuje ją do operatora
([standard: lex.ext](https://eel.is/c++draft/lex.ext#3),
[cppreference: Syntax](https://en.cppreference.com/cpp/language/user_literal#Syntax)).

```cpp
#include <cstdint>
#include <iostream>

struct Bytes {
    std::uint64_t value;
};

constexpr Bytes operator""_kib(unsigned long long n) { return Bytes{n * 1024}; }

int main() {
    std::cout << (16_kib).value << ' '        // 16384
              << (0x10_kib).value << ' '      // 16384 (hexadecimal 16)
              << (0b1'0000_kib).value << ' '  // 16384 (binary 16)
              << (1'024_kib).value << '\n';   // 1048576
}
```

## Literały w przestrzeni nazw `…::literals`

Literału nie da się poprzedzić nazwą przestrzeni (`mem::4_kib` nie istnieje): kompilator szuka operatora
zwykłym wyszukiwaniem nazw w miejscu użycia
([standard: lex.ext](https://eel.is/c++draft/lex.ext#2)). Dlatego biblioteki umieszczają operatory literałowe
w osobnej, zagnieżdżonej przestrzeni nazw `literals`, a użytkownik sam wprowadza ją dyrektywą `using namespace`
tam, gdzie chce z nich korzystać — dokładnie tak, jak robi to biblioteka standardowa ze `std::literals`
([Krzemieński, *User-defined literals — Part III*](https://akrzemi1.wordpress.com/2012/10/29/user-defined-literals-part-iii/)).
Ponieważ własne sufiksy zaczynają się od `_`, nie zderzą się z sufiksami standardowymi i można używać obu naraz
([C++ Core Guidelines SF.7](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#rs-using-directive)).

Typ z jednostką powinien też zapewniać operacje, które mają sens: tu porównanie (`==` generowane przez
`= default`) i dodawanie dwóch rozmiarów. Celowo brakuje niejawnej konwersji z liczby (`reserve(4096)` się nie
skompiluje) i mnożenia dwóch rozmiarów.

```cpp
#include <cstdint>
#include <iostream>

namespace mem {

struct Bytes {
    std::uint64_t value;
    friend constexpr bool operator==(Bytes, Bytes) = default;
};

constexpr Bytes operator+(Bytes a, Bytes b) { return Bytes{a.value + b.value}; }

namespace literals {
constexpr Bytes operator""_kib(unsigned long long n) { return Bytes{n * 1024}; }
}  // namespace literals

}  // namespace mem

void reserve(mem::Bytes size) { std::cout << "reserving " << size.value << " B\n"; }

int main() {
    using namespace mem::literals;

    reserve(4_kib);                       // reserving 4096 B
    reserve(1_kib + mem::Bytes{512});     // reserving 1536 B
    static_assert(2_kib == mem::Bytes{2048});
    // reserve(4096);  // error: could not convert '4096' from 'int' to 'mem::Bytes'
}
```

## Ćwiczenie 1 — odległości

*Treść ćwiczenia — w przygotowaniu.*

## Dozwolone listy parametrów

Operator literałowy nie może mieć dowolnych parametrów. Standard wymienia zamkniętą listę i **żadna inna nie
jest dozwolona** — ani `int`, ani `double`, ani `float`; parametry nie mogą też mieć wartości domyślnych
([standard: over.literal](https://eel.is/c++draft/over.literal#3),
[cppreference: Literal operators](https://en.cppreference.com/cpp/language/user_literal#Literal_operators)):

| Literał z sufiksem | Dozwolone parametry |
|---|---|
| całkowity: `12_x`, `0xFF_x` | `(unsigned long long)` |
| zmiennoprzecinkowy: `1.5_x`, `2e3_x` | `(long double)` |
| znakowy: `'a'_x` | `(char)`, `(wchar_t)`, `(char8_t)`, `(char16_t)`, `(char32_t)` |
| napisowy: `"abc"_x` | `(const char*, std::size_t)` i odpowiedniki dla `wchar_t`, `char8_t`, `char16_t`, `char32_t` |
| całkowity lub zmiennoprzecinkowy (forma „surowa”) | `(const char*)` albo szablon `template <char...>` — patrz notebook 03 |

Każde z tych przeciążeń może nosić ten sam sufiks. Parametr `std::size_t` występuje tylko w parze ze wskaźnikiem
na znaki — dostaje długość napisu. Spoza listy kompilator odrzuca nawet typ „prawie dobry”:

```cpp
struct Frequency {
    double hz;
};

Frequency operator""_ghz(float v);  // error: invalid literal operator parameter type 'float'
```

## Która forma zostanie wybrana?

Formę wybiera **rodzaj literału**, a nie jego wartość:

* literał całkowity (`5_x`, `0x10_x`, `1'000_x`) → operator z parametrem `unsigned long long`,
* literał zmiennoprzecinkowy (`2.4_x`, `1e9_x`) → operator z parametrem `long double`.

Między formami **nie ma konwersji**: jeśli istnieje tylko forma całkowita, literał z kropką się nie skompiluje —
i odwrotnie ([standard: lex.ext](https://eel.is/c++draft/lex.ext#4)). Halpern ujmuje to tak: przy literałach
użytkownika nie stosuje się promocji ani konwersji, argument musi pasować dokładnie
([slajdy ACCU 2022](https://accu.org/conf-docs/PDFs_2022/pablo_halpern_ACCU-2022-Embracing-User-Defined-Literals.pdf)).
Jeśli chcemy, by działały oba zapisy, piszemy dwa przeciążenia:

```cpp
#include <iostream>

struct Frequency {
    double hz;
};

constexpr Frequency operator""_ghz(unsigned long long v) {  // 5_ghz
    return Frequency{static_cast<double>(v) * 1e9};
}

constexpr Frequency operator""_ghz(long double v) {  // 2.4_ghz
    return Frequency{static_cast<double>(v) * 1e9};
}

int main() {
    Frequency wifi_low = 2.4_ghz;
    Frequency wifi_high = 5_ghz;
    std::cout << wifi_low.hz << ' ' << wifi_high.hz << '\n';  // 2.4e+09 5e+09
}
```

Bez drugiego przeciążenia `2.4_ghz` daje błąd kompilacji — GCC: `unable to find numeric literal operator
'operator""_ghz'`, Clang: `no matching literal operator for call to 'operator""_ghz' with argument of type
'long double' or 'const char *'`.

## Własne literały i `std::chrono`

Sufiksy `std::chrono` z notebooka 01 dają **różne typy**: `2h` to `std::chrono::hours`, `90s` —
`std::chrono::seconds`. Funkcja, która ma przyjąć dowolny czas trwania, jest więc szablonem względem
`std::chrono::duration<Rep, Period>`: `Rep` to typ licznika (np. `long`, `double`), `Period` — długość jednego
tyknięcia w sekundach (np. `std::ratio<3600>` dla godzin)
([C++ Core Guidelines I.4](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#ri-typed)).

Czas trwania z licznikiem zmiennoprzecinkowym (`std::chrono::duration<double, …>`) da się zbudować z każdego
innego czasu trwania bez `duration_cast`, bo nic nie jest obcinane. Licznik całkowity przyjmie tylko jednostkę,
którą da się zapisać dokładnie (minuty → sekundy, ale nie sekundy → minuty)
([cppreference: duration constructor](https://en.cppreference.com/cpp/chrono/duration/duration)).

```cpp
#include <chrono>
#include <iostream>
#include <ratio>

using namespace std::literals;

// accepts hours, minutes, seconds, milliseconds, ...
template <class Rep, class Period>
constexpr double to_hours(std::chrono::duration<Rep, Period> d) {
    return std::chrono::duration<double, std::ratio<3600>>{d}.count();
}

int main() {
    static_assert(to_hours(2h) == 2.0);
    std::cout << to_hours(90min) << ' ' << to_hours(1800s) << '\n';  // 1.5 0.5
}
```

Własne literały i literały `std::chrono` żyją obok siebie: wystarczą dwie dyrektywy
`using namespace std::literals;` i `using namespace <biblioteka>::literals;`.

## Ćwiczenie 2 — kąty

*Treść ćwiczenia — w przygotowaniu.*

## Ćwiczenie 3 — prędkość

*Treść ćwiczenia — w przygotowaniu.*

## Ciekawostki dla dociekliwych

### `-4_kib`: minus nie należy do literału

Literały w C++ nigdy nie są ujemne — minus to osobny operator jednoargumentowy, zastosowany do **wyniku**
operatora literałowego. Dlatego forma całkowita dostaje `unsigned long long`
([Krzemieński, *User-defined literals — Part I*](https://akrzemi1.wordpress.com/2012/08/12/user-defined-literals-part-i/)).
Jeśli typ wyniku nie ma operatora `-`, zapis `-4_kib` się nie skompiluje; jeśli wynik jest liczbą bez znaku,
minus „zawinie” wartość:

```cpp
#include <iostream>

constexpr unsigned long long operator""_u(unsigned long long v) { return v; }

int main() {
    std::cout << -5_u << '\n';  // 18446744073709551611: -(5ULL), not the literal -5
}
```

### Zachłanny lekser: `4s.count()`

Kompilator dzieli kod na tokeny, zawsze biorąc **najdłuższy** możliwy ciąg znaków (*maximal munch*). „Liczba
preprocesora” może zawierać litery, cyfry, `_` i kropki, więc `4s.count` jest jednym tokenem — a takiego sufiksu
nie ma ([cppreference: User-defined literals — Notes](https://en.cppreference.com/cpp/language/user_literal#Notes),
[cppreference: Maximal munch](https://en.cppreference.com/cpp/language/translation_phases#Maximal_munch)).
To samo dotyczy własnych literałów (`4_kib.value`). Pomaga nawias albo spacja: `(4s).count()`, `4s .count()`.

```cpp
#include <chrono>

using namespace std::literals;

auto ticks = 4s.count();  // GCC: unable to find numeric literal operator 'operator""s.count'
```

### `0x1e_x` to 30, a nie wykładnik

W literale szesnastkowym `e` jest cyfrą (14), więc `0x1e_x` przekazuje do operatora wartość 30. Pułapka pojawia
się, gdy po takim `e` stoi znak: `0xe+1` jest dla leksera **jednym** tokenem (`e+` wygląda jak początek
wykładnika), więc trzeba pisać `0xe + 1`
([standard: lex.pptoken, przykład 2](https://eel.is/c++draft/lex.pptoken#example-2)).

```cpp
#include <iostream>

constexpr unsigned long long operator""_x(unsigned long long v) { return v; }

int main() {
    std::cout << 0x1e_x << ' ' << 0xe + 1 << '\n';  // 30 15
}
```

## Linki i materiały

* [cppreference: User-defined literals](https://en.cppreference.com/cpp/language/user_literal#Literal_operators) — składnia, dozwolone parametry, uwagi
* [Standard C++ (szkic): lex.ext](https://eel.is/c++draft/lex.ext) i [over.literal](https://eel.is/c++draft/over.literal) — reguły wyboru operatora i dozwolone listy parametrów
* [Microsoft Learn: User-defined literals](https://learn.microsoft.com/en-us/cpp/cpp/user-defined-literals-cpp#user-defined-literal-operator-signatures)
  — po polsku: [Literały zdefiniowane przez użytkownika](https://learn.microsoft.com/pl-pl/cpp/cpp/user-defined-literals-cpp#user-defined-literal-operator-signatures)
  (tłumaczenie maszynowe). Uwaga: strona używa przestarzałego zapisu ze spacją `operator "" _x`, nie zna `char8_t`
  i błędnie podaje `long long` zamiast `unsigned long long`.
* [Andrzej Krzemieński: User-defined literals — Part I](https://akrzemi1.wordpress.com/2012/08/12/user-defined-literals-part-i/)
  i [Part III](https://akrzemi1.wordpress.com/2012/10/29/user-defined-literals-part-iii/) — motywacja i przestrzenie nazw
  (tekst z 2012 r.: zapis ze spacją po `""` był wtedy wymagany, dziś jest przestarzały)
* [Pablo Halpern: Embracing User Defined Literals Safely (ACCU 2022, slajdy)](https://accu.org/conf-docs/PDFs_2022/pablo_halpern_ACCU-2022-Embracing-User-Defined-Literals.pdf)
* [Jonathan Boccara: Strong types for strong interfaces (Meeting C++ 2017, wideo)](https://www.youtube.com/watch?v=WVleZqzTw2k)
* [Mateusz Pusz: A Physical Units Library For the Next C++ (CppCon 2020, wideo)](https://www.youtube.com/watch?v=7dExYGSOJzo)
  — biblioteka jednostek fizycznych; jej [FAQ](https://mpusz.github.io/mp-units/latest/getting_started/faq/#why-dont-we-use-udls-to-create-quantities)
  wyjaśnia, dlaczego zrezygnowano w niej z literałów użytkownika

---

[Strona główna](../_start.ipynb)